<a href="https://colab.research.google.com/github/Thanaphon-673020253-2/project_bigdata/blob/main/PART3_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# Part 3: Data Collection Robot + External Business Insight *(งานกลุ่ม — 25 คะแนน)*

ส่วนนี้ให้ทีมลอง **สร้าง dataset เล็ก ๆ ของตัวเอง** จากข้อมูลออนไลน์ แล้วใช้ตอบคำถามหนึ่งข้อ

เลือกวิธีเก็บเพียง **1 วิธี**
* API
* RSS
* Web Robot / Crawler / Scraper

## สิ่งที่ต้องทำ

1. ตั้งคำถามว่า **ข้อมูลนี้จะช่วยตัดสินใจเรื่องอะไร**
2. นิยามก่อนว่า **1 record คืออะไร** เช่น 1 ข่าว / 1 post / 1 complaint
3. เก็บอย่างน้อย **200 valid analytical records หลัง clean**
4. ตรวจ duplicate, missing และ record ที่ใช้ไม่ได้
5. ใช้เทคนิควิเคราะห์ที่เหมาะสม **1–2 เทคนิคก็เพียงพอ**
6. สรุป **1 Insight Card** พร้อมข้อจำกัด

**ข้อกำหนดด้านความรับผิดชอบ**
* เคารพ `robots.txt`, rate limit และ Terms of Service
* ใส่ delay เมื่อดึงหน้าเว็บ
* ห้าม bypass login, CAPTCHA หรือ paywall
* ไม่เก็บข้อมูลส่วนบุคคลที่ไม่จำเป็น

> Part 3 ไม่ได้วัดว่าใคร scrape ได้เยอะที่สุด แต่วัดว่าใครสร้างข้อมูลที่ใช้วิเคราะห์ได้จริงและตอบคำถามได้


### 3.1 เลือกแหล่งข้อมูลและวิธีเก็บ

เลือกแหล่งข้อมูลที่สัมพันธ์กับคำถามของทีม แล้วใช้ **เพียง 1 วิธีเก็บ**

ตัวอย่าง:
* Hacker News API — post + score + comments
* RSS — ข่าว/บทความ
* Traffy Fondue Open Data — เรื่องร้องเรียน
* เว็บไซต์สาธารณะที่อนุญาต crawler — ข่าว/บทความ/รายการต่าง ๆ

**กรอกก่อนเขียนโค้ด**
* แหล่งข้อมูล: ...
* คำถาม: ...
* ผู้ใช้ผลวิเคราะห์: ...
* **1 record คือ:** ...
* วิธีเก็บ: API / RSS / Robot
* field สำคัญที่จะเก็บ: ...

ด้านล่างมี
* **ตัวอย่างที่ 1:** API แบบสั้น
* **ตัวอย่างที่ 2–3:** ทางเลือกอื่นแบบสรุป
* **ตัวอย่างที่ 4:** Web Robot แบบอธิบายทีละขั้น


* # Source : The Movie DB
* # Question :
* # Person Analytic : platform streaming
* # 1 record  : 1 review


# **3.1 เตรียมข้อมูล**
* # ไม่ต้องรันแล้ว ใช้ไฟล์ที่โหลดเอานะ

In [ ]:
# ===== Setup =====
import os, json, time
from getpass import getpass
import numpy as np
import pandas as pd
import requests

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ---- โฟลเดอร์และไฟล์ ----
DATA_DIR      = "./data_part3/"
os.makedirs(DATA_DIR, exist_ok=True)
TV_RAW_PATH   = DATA_DIR + "tmdb_tv_raw.jsonl"
REV_RAW_PATH  = DATA_DIR + "tmdb_reviews_raw.jsonl"
LOG_PATH      = DATA_DIR + "collection_log.json"
CORPUS_JSONL  = DATA_DIR + "tmdb_corpus_clean.jsonl"
CORPUS_CSV    = DATA_DIR + "tmdb_corpus_clean.csv"

# ---- สวิตช์: False = ถ้ามีไฟล์ raw อยู่แล้วให้ใช้ไฟล์เดิม (ข้อมูลไม่เปลี่ยน) ----
FORCE_REFETCH = False

# ---- พารามิเตอร์การเก็บข้อมูล ----
LANGUAGE         = "en-US"
SORT_BY          = "popularity.desc"
MAX_PAGES        = 30        # 1 หน้า = 20 เรื่อง -> ~600 เรื่อง
MAX_REVIEW_PAGES = 30        # 1 หน้า <= 20 รีวิว/เรื่อง (หยุดเองเมื่อหมดหน้า)
DELAY            = 0.1       # วินาที ระหว่าง request

print("ข้อมูลจะถูกบันทึกที่:", os.path.abspath(DATA_DIR))

ข้อมูลจะถูกบันทึกที่: /content/data_part3


# ฟังก์ชันที่ใช้เก็บข้อมูล
ยังไม่เรียก API ในเซลล์นี้ (token จะถูกขอเมื่อมีการเรียกครั้งแรกเท่านั้น)

# --------------------------**ห้ามรันตั้งแต่ส่วนนี้น้า** ---------------------------

In [ ]:
# ===== A0: ฟังก์ชัน =====
BASE_URL = "https://api.themoviedb.org/3"
session = None

def get_session():
    """ใช้ "API Read Access Token" (ขึ้นต้น eyJ...) ไม่ใช่ API Key 32 ตัว
    Colab: เก็บเป็น Secret ชื่อ TMDB_TOKEN | ที่อื่น: ตัวแปรสภาพแวดล้อม TMDB_TOKEN หรือพิมพ์ตอนรัน
    ห้ามเขียน token ลงใน notebook"""
    try:
        from google.colab import userdata
        token = userdata.get("TMDB_TOKEN")
    except Exception:
        token = os.environ.get("TMDB_TOKEN") or getpass("วาง TMDB API Read Access Token: ")
    s = requests.Session()
    s.headers.update({"Authorization": f"Bearer {token}", "accept": "application/json"})
    return s

def tmdb_get(path, params=None, retries=4):
    """เรียก TMDB พร้อม retry เมื่อโดน rate limit (429) หรือ server error (5xx)"""
    global session
    if session is None:
        session = get_session()
    for attempt in range(retries):
        r = session.get(BASE_URL + path, params=params, timeout=20)
        if r.status_code == 429:
            wait = int(r.headers.get("Retry-After", 2))
            print(f"โดน rate limit รอ {wait}s ...")
            time.sleep(wait)
            continue
        if r.status_code >= 500:
            time.sleep(2 * (attempt + 1))
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError(f"เรียก {path} ไม่สำเร็จหลัง {retries} ครั้ง")

def load_jsonl(path):
    """อ่าน JSON Lines ทีละบรรทัด (ถ้าไฟล์เสียจะบอกเลขบรรทัด)"""
    rows = []
    with open(path, encoding="utf-8") as f:
        for n, line in enumerate(f, 1):
            if line.strip():
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as e:
                    raise ValueError(f"{path} บรรทัด {n} อ่านไม่ได้: {e}")
    return pd.DataFrame(rows)

def save_jsonl(df, path):
    df.to_json(path, orient="records", lines=True, force_ascii=False, date_format="iso")

def load_log():
    return json.load(open(LOG_PATH, encoding="utf-8")) if os.path.exists(LOG_PATH) else {}

def save_log(log):
    json.dump(log, open(LOG_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)

# เก็บซีรีส์ (/discover/tv → /tv/{id})
ถ้ามี tmdb_tv_raw.jsonl อยู่แล้วและ FORCE_REFETCH = False จะอ่านไฟล์เดิม ไม่เรียก API

In [ ]:
# ===== A1: ซีรีส์ =====
def fetch_discover(max_pages=MAX_PAGES):
    rows = []
    for page in range(1, max_pages + 1):
        data = tmdb_get("/discover/tv", {"language": LANGUAGE, "sort_by": SORT_BY,
                                         "include_adult": "false", "page": page})
        results = data.get("results", [])
        if not results:
            break
        rows.extend(results)
        if page % 10 == 0:
            print(f"ดึงรายการแล้ว {page} หน้า / {len(rows)} เรื่อง")
        if page >= data.get("total_pages", page):
            break
        time.sleep(DELAY)
    return rows

def build_tv_record(d, item, genre_map):
    tv_id = d["id"]
    return {
        "doc_id": str(tv_id),
        "name": d.get("name"),
        "original_name": d.get("original_name"),
        "original_language": d.get("original_language"),
        "origin_country": "|".join(d.get("origin_country") or []),
        "first_air_date": d.get("first_air_date"),
        "last_air_date": d.get("last_air_date"),
        "genres": "|".join(g["name"] for g in d.get("genres", [])) or "|".join(genre_map.get(x, str(x)) for x in item.get("genre_ids", [])),
        "overview": d.get("overview") or item.get("overview") or "",
        "vote_average": d.get("vote_average"),
        "vote_count": d.get("vote_count"),
        "popularity": d.get("popularity"),
        "number_of_seasons": d.get("number_of_seasons"),
        "number_of_episodes": d.get("number_of_episodes"),
        "status": d.get("status"),
        "in_production": d.get("in_production"),
        "source": "tmdb",
        "url": f"https://www.themoviedb.org/tv/{tv_id}",
    }

if os.path.exists(TV_RAW_PATH) and not FORCE_REFETCH:
    print("พบ", TV_RAW_PATH, "-> ใช้ไฟล์เดิม ไม่เรียก API")
    df_tv_raw = load_jsonl(TV_RAW_PATH)
    df_tv_raw["doc_id"] = df_tv_raw["doc_id"].astype(str)
    log = load_log()
else:
    genre_map = {g["id"]: g["name"] for g in tmdb_get("/genre/tv/list", {"language": LANGUAGE})["genres"]}
    list_rows = fetch_discover()

    seen, uniq = set(), []                         # /discover/tv อาจส่งซีรีส์เดิมซ้ำข้ามหน้า -> ตัดก่อนเรียก detail
    for it in list_rows:
        if it["id"] not in seen:
            seen.add(it["id"]); uniq.append(it)

    records, failed = [], []
    for i, item in enumerate(uniq, start=1):
        try:
            records.append(build_tv_record(tmdb_get(f"/tv/{item['id']}", {"language": LANGUAGE}), item, genre_map))
        except Exception as e:
            failed.append((item["id"], str(e)))
        if i % 100 == 0:
            print(f"detail {i}/{len(uniq)}")
        time.sleep(DELAY)

    df_tv_raw = pd.DataFrame(records)
    save_jsonl(df_tv_raw, TV_RAW_PATH)
    log = {"series_listed": len(list_rows), "series_list_duplicates": len(list_rows) - len(uniq),
           "series_detail_ok": len(records), "series_detail_failed": len(failed)}
    save_log(log)
    print(f"✅ ซีรีส์: รายการ {len(list_rows)} -> ไม่ซ้ำ {len(uniq)} -> detail สำเร็จ {len(records)} / ล้มเหลว {len(failed)}")

print("ซีรีส์ดิบ:", len(df_tv_raw), "เรื่อง")

ดึงรายการแล้ว 10 หน้า / 200 เรื่อง
ดึงรายการแล้ว 20 หน้า / 400 เรื่อง
ดึงรายการแล้ว 30 หน้า / 600 เรื่อง
detail 100/596
detail 200/596
detail 300/596
detail 400/596
detail 500/596
✅ ซีรีส์: รายการ 600 -> ไม่ซ้ำ 596 -> detail สำเร็จ 596 / ล้มเหลว 0
ซีรีส์ดิบ: 596 เรื่อง


# เก็บรีวิว (/tv/{id}/reviews)
วนทีละซีรีส์ ไม่ซ้ำ id (ไม่วนตามตารางที่มี id ซ้ำ) และไม่เก็บชื่อผู้รีวิว

In [ ]:
MAX_REVIEW_PAGES_LOCAL = MAX_REVIEW_PAGES

def fetch_reviews(tv_id, max_pages=MAX_REVIEW_PAGES_LOCAL):
    out = []
    for page in range(1, max_pages + 1):
        data = tmdb_get(f"/tv/{tv_id}/reviews", {"page": page})
        for r in data.get("results", []):
            out.append({
                "review_id": r.get("id"),
                "doc_id": str(tv_id),
                "text": r.get("content") or "",
                "user_rating": (r.get("author_details") or {}).get("rating"),   # อาจเป็น None
                "created_at": r.get("created_at"),
                # ไม่เก็บ author/username เพื่อลดข้อมูลส่วนบุคคล (PDPA)
            })
        if page >= data.get("total_pages", 1):
            break
        time.sleep(DELAY)
    return out

if os.path.exists(REV_RAW_PATH) and not FORCE_REFETCH:
    print("พบ", REV_RAW_PATH, "-> ใช้ไฟล์เดิม ไม่เรียก API")
    df_rev_raw = load_jsonl(REV_RAW_PATH)
    df_rev_raw["doc_id"] = df_rev_raw["doc_id"].astype(str)
else:
    ids = df_tv_raw["doc_id"].drop_duplicates().tolist()
    rev_rows, rev_failed = [], []
    for i, tv_id in enumerate(ids, start=1):
        try:
            rev_rows.extend(fetch_reviews(tv_id))
        except Exception as e:
            rev_failed.append(tv_id)
        if i % 100 == 0:
            print(f"{i}/{len(ids)} เรื่อง | รีวิวสะสม {len(rev_rows)}")
        time.sleep(DELAY)

    n_first_fail = len(rev_failed)                  # ลองใหม่อีก 1 รอบสำหรับเรื่องที่ล้มเหลว
    still_failed = []
    for tv_id in rev_failed:
        try:
            rev_rows.extend(fetch_reviews(tv_id))
        except Exception:
            still_failed.append(tv_id)
        time.sleep(DELAY)

    df_rev_raw = pd.DataFrame(rev_rows)
    save_jsonl(df_rev_raw, REV_RAW_PATH)
    log.update({"review_series_requested": len(ids), "review_series_failed_first_try": n_first_fail,
                "review_series_failed_final": len(still_failed), "review_rows_raw": len(df_rev_raw)})
    save_log(log)
    print(f"✅ รีวิว: ขอจาก {len(ids)} เรื่อง -> ได้ {len(df_rev_raw)} แถว | ล้มเหลวรอบแรก {n_first_fail} / หลังลองใหม่ {len(still_failed)}")

print("รีวิวดิบ:", len(df_rev_raw), "แถว")

100/596 เรื่อง | รีวิวสะสม 144
200/596 เรื่อง | รีวิวสะสม 273
300/596 เรื่อง | รีวิวสะสม 335
400/596 เรื่อง | รีวิวสะสม 408
500/596 เรื่อง | รีวิวสะสม 490
✅ รีวิว: ขอจาก 596 เรื่อง -> ได้ 576 แถว | ล้มเหลวรอบแรก 0 / หลังลองใหม่ 0
รีวิวดิบ: 576 แถว


# --------------------------**ห้ามรันจนถึงส่วนนี้น้า** ---------------------------

# **3.2 ตรวจคุณภาพข้อมูล (Clean + Quality Report)**
ลำดับ: Clean ซีรีส์ → Clean รีวิว → Join → รายงาน → บันทึก final corpus
ทุกเซลล์ในส่วนนี้ใช้เฉพาะไฟล์ raw ที่โหลดแล้ว จึงรันซ้ำได้โดยผลเดิม

In [ ]:
# ===== 3.2 ตรวจคุณภาพข้อมูล =====
# ===== อ่านข้อมูล Raw จากไฟล์ =====

import pandas as pd

df_raw = pd.read_json(
    "tmdb_tv_raw.jsonl",
    lines=True
)                                                           # อ่านข้อมูล TV Series ดิบ

df_rev_raw = pd.read_json(
    "tmdb_reviews_raw.jsonl",
    lines=True
)                                                           # อ่านข้อมูล Reviews ดิบ

print("TV Series Raw :", len(df_raw), "records")
print("Reviews Raw   :", len(df_rev_raw), "records")

TV Series Raw : 596 records
Reviews Raw   : 576 records


In [ ]:
# ===== Clean ข้อมูล TV Series =====

tv_clean = df_raw.copy()                                    # สำเนาข้อมูล TV Series ดิบ
tv_before = len(tv_clean)                                   # จำนวนข้อมูลก่อน Clean

tv_duplicate = tv_clean.duplicated(
    subset=["doc_id"], keep="first"
)                                                           # ตรวจ doc_id ที่ซ้ำ
n_tv_duplicate = tv_duplicate.sum()
tv_clean = tv_clean[~tv_duplicate].copy()                   # ตัดข้อมูลซ้ำออก

tv_missing_id = (
    tv_clean["doc_id"].isna() |
    tv_clean["doc_id"].astype(str).str.strip().eq("")
)                                                           # ตรวจ doc_id ที่หายไป

n_tv_missing_id = tv_missing_id.sum()
tv_clean = tv_clean[~tv_missing_id].copy()                  # ตัด record ที่ไม่มี doc_id

tv_clean["first_air_date"] = pd.to_datetime(
    tv_clean["first_air_date"], errors="coerce"
)                                                           # แปลงวันที่เป็น datetime

tv_clean["last_air_date"] = pd.to_datetime(
    tv_clean["last_air_date"], errors="coerce"
)                                                           # แปลงวันที่เป็น datetime

tv_clean = tv_clean.reset_index(drop=True)                  # จัด index ใหม่

print("===== TV SERIES CLEANING =====")
print("ก่อน Clean        :", tv_before)
print("Duplicate         :", n_tv_duplicate)
print("Missing doc_id    :", n_tv_missing_id)
print("หลัง Clean         :", len(tv_clean))

===== TV SERIES CLEANING =====
ก่อน Clean        : 596
Duplicate         : 0
Missing doc_id    : 0
หลัง Clean         : 596


In [ ]:
# ===== Clean ข้อมูล Reviews =====

rev_clean = df_rev_raw.copy()                                # สำเนาข้อมูล Reviews ดิบ
rev_before = len(rev_clean)                                  # จำนวน Reviews ก่อน Clean

rev_duplicate = rev_clean.duplicated(
    subset=["review_id"], keep="first"
)                                                           # ตรวจ review_id ที่ซ้ำ

n_rev_duplicate = rev_duplicate.sum()
rev_clean = rev_clean[~rev_duplicate].copy()                 # ตัด Reviews ที่ซ้ำ

rev_clean["text"] = rev_clean["text"].fillna("").str.strip() # จัดการ text ที่เป็น missing
missing_text = rev_clean["text"].eq("")                      # ตรวจข้อความที่ว่าง
n_rev_missing = missing_text.sum()
rev_clean = rev_clean[~missing_text].copy()                  # ตัด Reviews ที่ไม่มีข้อความ

rev_clean["text_length"] = rev_clean["text"].str.len()       # นับจำนวนตัวอักษรของแต่ละ Review

print("===== REVIEW QUALITY CHECK =====")
print("ก่อน Clean          :", rev_before)
print("Duplicate           :", n_rev_duplicate)
print("Missing text        :", n_rev_missing)

print("\nความยาว Review:")
print(rev_clean["text_length"].describe())

===== REVIEW QUALITY CHECK =====
ก่อน Clean          : 576
Duplicate           : 0
Missing text        : 0

ความยาว Review:
count      576.000000
mean      1153.302083
std       2486.228158
min          4.000000
25%        273.000000
50%        578.000000
75%       1097.500000
max      33616.000000
Name: text_length, dtype: float64


In [ ]:
# ===== ตัด Reviews ที่สั้นผิดปกติ =====

MIN_REVIEW_LENGTH = 50                                      # กำหนดข้อความสั้นผิดปกติน้อยกว่า 50 ตัวอักษร

short_mask = rev_clean["text_length"] < MIN_REVIEW_LENGTH   # ตรวจ Review ที่สั้นผิดปกติ
n_rev_short = short_mask.sum()

rev_clean = rev_clean[~short_mask].copy()                   # ตัด Review ที่สั้นผิดปกติ
rev_clean = rev_clean.reset_index(drop=True)                # จัด index ใหม่

print("Review สั้นกว่า", MIN_REVIEW_LENGTH, "ตัวอักษร =", n_rev_short)
print("Reviews หลัง Clean =", len(rev_clean))

Review สั้นกว่า 50 ตัวอักษร = 24
Reviews หลัง Clean = 552


In [ ]:
# ===== Join TV Series กับ Reviews =====

corp = rev_clean.merge(
    tv_clean,
    on="doc_id",
    how="inner"
)                                                           # เชื่อม Reviews กับ TV Series ด้วย doc_id

print("Reviews ก่อน Join =", len(rev_clean))
print("Final corpus หลัง Join =", len(corp))
print("จำนวนซีรีส์ใน corpus =", corp["doc_id"].nunique())

Reviews ก่อน Join = 552
Final corpus หลัง Join = 552
จำนวนซีรีส์ใน corpus = 204


In [ ]:
# ===== Final Data Quality Report =====

n_join_removed = len(rev_clean) - len(corp)                 # จำนวน Review ที่ Join ไม่ได้

print("===== DATA QUALITY REPORT =====")
print(f"Reviews ที่เก็บได้ทั้งหมด       : {rev_before} records")
print(f"Duplicate                       : {n_rev_duplicate}")
print(f"Missing text                    : {n_rev_missing}")
print(f"ข้อความสั้นกว่า {MIN_REVIEW_LENGTH} ตัวอักษร   : {n_rev_short}")
print(f"Join กับ TV Series ไม่ได้       : {n_join_removed}")

print("\n===== FINAL CORPUS =====")
print(f"Final corpus                    : {len(corp)} records")
print(f"จำนวนซีรีส์                     : {corp['doc_id'].nunique()} เรื่อง")

print("\nช่วงเวลาของ Reviews:")
print(
    pd.to_datetime(corp["created_at"], errors="coerce").min(),
    "ถึง",
    pd.to_datetime(corp["created_at"], errors="coerce").max()
)

genres_covered = sorted({
    genre
    for genres in corp["genres"].dropna()
    for genre in genres.split("|")
    if genre
})                                                          # ตรวจ Genres ที่ corpus ครอบคลุม

print("\nGenres ที่ครอบคลุม:")
print(genres_covered)
print("จำนวน Genres =", len(genres_covered))

if len(corp) >= 200:
    print("\nผ่านเกณฑ์: มีอย่างน้อย 200 valid analytical records")
else:
    print("\nยังไม่ผ่านเกณฑ์: มีน้อยกว่า 200 valid analytical records")

===== DATA QUALITY REPORT =====
Reviews ที่เก็บได้ทั้งหมด       : 576 records
Duplicate                       : 0
Missing text                    : 0
ข้อความสั้นกว่า 50 ตัวอักษร   : 24
Join กับ TV Series ไม่ได้       : 0

===== FINAL CORPUS =====
Final corpus                    : 552 records
จำนวนซีรีส์                     : 204 เรื่อง

ช่วงเวลาของ Reviews:
2017-02-11 21:46:06+00:00 ถึง 2026-09-29 21:26:10+00:00

Genres ที่ครอบคลุม:
['Action & Adventure', 'Animation', 'Comedy', 'Crime', 'Drama', 'Family', 'Kids', 'Mystery', 'News', 'Reality', 'Sci-Fi & Fantasy', 'Soap', 'Talk', 'War & Politics', 'Western']
จำนวน Genres = 15

ผ่านเกณฑ์: มีอย่างน้อย 200 valid analytical records


In [ ]:
# ===== บันทึก Final corpus =====

jsonl_path = "tmdb_corpus_clean.jsonl"                      # Final corpus แบบ JSONL
csv_path = "tmdb_corpus_clean.csv"                          # Final corpus แบบ CSV

corp.to_json(
    jsonl_path,
    orient="records",
    lines=True,
    force_ascii=False,
    date_format="iso"
)

corp.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", jsonl_path)
print("Saved:", csv_path)

corp_check = pd.read_json(jsonl_path, lines=True)            # ตรวจว่าไฟล์อ่านกลับได้
print("อ่านกลับได้ =", len(corp_check), "records")

Saved: tmdb_corpus_clean.jsonl
Saved: tmdb_corpus_clean.csv
อ่านกลับได้ = 552 records


# ---------------**ส่วนข้างบนนี้ก็ไม่ต้องรันแล้วไปโหลดในไดร์ฟแล้วอ่านไฟล์เอานะ**--------------

### 3.3 หา Insight

ใช้ **1–2 เทคนิค** ที่เหมาะกับคำถาม เช่น

* frequency / n-gram
* sentiment
* topic
* comparison ระหว่างกลุ่ม
* trend ตามเวลา

สร้างกราฟหรือตารางหลักอย่างน้อย **1 ชิ้น** แล้วตอบว่า

> “จากข้อมูลนี้ ผู้ใช้ผลวิเคราะห์ควรรู้อะไรหรือทำอะไรต่อ?”


In [25]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    CountVectorizer,
    ENGLISH_STOP_WORDS
)
from sklearn.decomposition import NMF


# Load dataset
df = pd.read_csv("tmdb_corpus_clean.csv")

print("Dataset shape:", df.shape)

display(df.head())


Dataset shape: (552, 23)


,review_id,doc_id,text,user_rating,created_at,text_length,name,original_name,original_language,origin_country,...,overview,vote_average,vote_count,popularity,number_of_seasons,number_of_episodes,status,in_production,source,url
0,66c1325eecbb1915cc898af9,2734,This series is a poster child for the way the ...,3.0,2024-08-17 23:29:34+00:00,1118,Law & Order: Special Victims Unit,Law & Order: Special Victims Unit,en,US,...,"In the criminal justice system, sexually-based...",7.956,4362,566.1656,28,598,Returning Series,True,tmdb,https://www.themoviedb.org/tv/2734
1,6708dd1b8d487f6541286af6,2734,**26 SEASONS** there's a reason SVU is the lon...,9.0,2024-10-11 08:08:59+00:00,403,Law & Order: Special Victims Unit,Law & Order: Special Victims Unit,en,US,...,"In the criminal justice system, sexually-based...",7.956,4362,566.1656,28,598,Returning Series,True,tmdb,https://www.themoviedb.org/tv/2734
2,65ec82636f31af0185e83fb6,59941,"Fallon. What is there to say? He's not funny, ...",2.0,2024-03-09 15:38:11+00:00,265,The Tonight Show Starring Jimmy Fallon,The Tonight Show Starring Jimmy Fallon,en,US,...,After Jay Leno's second retirement from the pr...,5.777,391,587.7775,14,2421,Returning Series,True,tmdb,https://www.themoviedb.org/tv/59941
3,63e74c8063aad200e05c90d5,108978,Bought Reacher: Season 1 on Blu-Ray. I absolut...,NaN,2023-02-11 08:06:24+00:00,607,Reacher,Reacher,en,US,...,"Jack Reacher, a veteran military police invest...",8.134,3436,603.2777,4,32,Returning Series,True,tmdb,https://www.themoviedb.org/tv/108978
4,65ab61170c4c1600c57bd9ce,108978,Season one was a 10/10 show. I LOVED it.\r\nI ...,8.0,2024-01-20 05:58:47+00:00,349,Reacher,Reacher,en,US,...,"Jack Reacher, a veteran military police invest...",8.134,3436,603.2777,4,32,Returning Series,True,tmdb,https://www.themoviedb.org/tv/108978


In [69]:
# 1. IMPORT LIBRARIES
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfVectorizer,
    ENGLISH_STOP_WORDS
)

from sklearn.decomposition import NMF
# 3. CHECK REQUIRED COLUMNS

required_columns = [
    "name",
    "popularity",
    "user_rating",
    "text"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"ไม่พบคอลัมน์: {missing_columns}"
    )

# 4. DATA PREPARATION
df["popularity"] = pd.to_numeric(
    df["popularity"],
    errors="coerce"
)

df["user_rating"] = pd.to_numeric(
    df["user_rating"],
    errors="coerce"
)

df["text"] = df["text"].fillna("")

df = df.dropna(
    subset=[
        "popularity",
        "user_rating"
    ]
).copy()
# 5. FIND TOP 20% POPULARITY
popularity_cutoff = df["popularity"].quantile(0.80)

high_popularity = df[
    df["popularity"] >= popularity_cutoff
].copy()


print("\n" + "=" * 70)
print("HIGH-POPULARITY THRESHOLD")
print("=" * 70)

print(
    f"80th Percentile Popularity = "
    f"{popularity_cutoff:.4f}"
)

print(
    f"High-popularity reviews = "
    f"{len(high_popularity)}"
)

# 6. SELECT NEGATIVE REVIEWS

negative_reviews = high_popularity[
    high_popularity["user_rating"] <= 5
].copy()


print("\n" + "=" * 70)
print("HIGH-POPULARITY SERIES WITH NEGATIVE REVIEWS")
print("=" * 70)

print(
    f"Negative reviews = "
    f"{len(negative_reviews)}"
)

print(
    f"Number of series = "
    f"{negative_reviews['name'].nunique()}"
)


display(
    negative_reviews[
        [
            "name",
            "popularity",
            "user_rating",
            "text"
        ]
    ].head(10)
)
# 7. STOPWORDS
custom_stopwords = {
    # General words
    "show",
    "series",
    "season",
    "episode",
    "episodes",
    "movie",
    "watch",
    "watched",
    "watching",
    "thing",
    "things",
    "really",
    "just",
    "like",
    "would",
    "could",
    "also",

    # Contraction fragments
    "don",
    "doesn",
    "didn",
    "isn",
    "wasn",
    "weren",
    "wouldn",
    "couldn",
    "shouldn",
    "hasn",
    "haven",
    "hadn",

    # General verbs
    "get",
    "got",
    "getting",
    "make",
    "made",
    "making",
    "say",
    "said",
    "says",
    "tell",
    "told",

    # Preprocessing fragments
    "ing"
}

all_stopwords = (
    set(ENGLISH_STOP_WORDS)
    .union(custom_stopwords)
)
# 8. EXPAND CONTRACTIONS

def expand_contractions(text):

    text = str(text).lower()

    contractions = {

        "don't": "do not",
        "doesn't": "does not",
        "didn't": "did not",

        "isn't": "is not",
        "wasn't": "was not",
        "weren't": "were not",

        "wouldn't": "would not",
        "couldn't": "could not",
        "shouldn't": "should not",

        "can't": "cannot",

        "could've": "could have",
        "would've": "would have",
        "should've": "should have",

        "it's": "it is",
        "that's": "that is",
        "there's": "there is",

        "they're": "they are",
        "you're": "you are",
        "we're": "we are",

        "i'm": "i am",
        "i've": "i have",
        "i'll": "i will",
        "i'd": "i would"
    }

    for contraction, replacement in contractions.items():

        text = text.replace(
            contraction,
            replacement
        )

    return text
# 9. CLEAN REVIEW TEXT

def clean_review(text):

    text = expand_contractions(text)

    # ----------------------------------------
    # Remove series names
    # ----------------------------------------

    for series_name in (
        negative_reviews["name"]
        .dropna()
        .unique()
    ):

        series_name = (
            str(series_name)
            .lower()
            .strip()
        )

        if series_name:

            text = re.sub(
                re.escape(series_name),
                " ",
                text,
                flags=re.IGNORECASE
            )

    # ----------------------------------------
    # Remove URLs
    # ----------------------------------------

    text = re.sub(
        r"http\S+|www\S+",
        " ",
        text
    )

    # ----------------------------------------
    # Keep English letters only
    # ----------------------------------------

    text = re.sub(
        r"[^a-zA-Z\s]",
        " ",
        text
    )

    # ----------------------------------------
    # Tokenize
    # ----------------------------------------

    words = text.lower().split()

    # ----------------------------------------
    # Remove stopwords / short words
    # ----------------------------------------

    words = [
        word
        for word in words
        if (
            word not in all_stopwords
            and len(word) >= 3
        )
    ]

    return " ".join(words)
# 10. APPLY CLEANING

negative_reviews["clean_text"] = (
    negative_reviews["text"]
    .fillna("")
    .apply(clean_review)
)


print("\n" + "=" * 70)
print("CLEANING CHECK")
print("=" * 70)

display(
    negative_reviews[
        [
            "name",
            "user_rating",
            "clean_text"
        ]
    ].head(10)
)

# 11. REMOVE EMPTY REVIEWS
negative_reviews = negative_reviews[
    negative_reviews["clean_text"].str.strip() != ""
].copy()


if len(negative_reviews) < 4:

    raise ValueError(
        "จำนวน Review หลัง Cleaning น้อยเกินไป "
        "ไม่เพียงพอสำหรับ Topic Modeling"
    )
# 12. KEYWORD / N-GRAM
keyword_vectorizer = CountVectorizer(

    ngram_range=(1, 2),

    min_df=2,

    max_features=3000,

    token_pattern=r"(?u)\b[a-zA-Z]{3,}\b"
)


X = keyword_vectorizer.fit_transform(
    negative_reviews["clean_text"]
)

terms = (
    keyword_vectorizer
    .get_feature_names_out()
)
# 13. TOP KEYWORDS
keyword_frequency = (
    (X > 0)
    .sum(axis=0)
    .A1
)


keyword_df = pd.DataFrame({

    "Keyword": terms,

    "Frequency": keyword_frequency

})


keyword_df = (
    keyword_df
    .sort_values(
        "Frequency",
        ascending=False
    )
    .head(20)
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print("TOP 20 KEYWORDS / N-GRAMS")
print("=" * 70)

display(keyword_df)
# 14. CROSS-SERIES COMMON CRITICISM

X_binary = (
    X > 0
).astype(int)


cross_series = []


for i, term in enumerate(terms):

    review_indices = (
        X_binary[:, i]
        .toarray()
        .flatten()
        > 0
    )

    selected_reviews = (
        negative_reviews
        .loc[review_indices]
    )

    frequency = len(
        selected_reviews
    )

    number_of_series = (
        selected_reviews["name"]
        .nunique()
    )

    cross_series.append({

        "Keyword": term,

        "Frequency": frequency,

        "Number_of_Series":
            number_of_series
    })


cross_series_df = pd.DataFrame(
    cross_series
)


cross_series_df = (
    cross_series_df
    .sort_values(

        [
            "Number_of_Series",
            "Frequency"
        ],

        ascending=False
    )
    .head(20)
    .reset_index(drop=True)
)


print("\n" + "=" * 70)
print("CROSS-SERIES COMMON CRITICISM")
print("=" * 70)

display(
    cross_series_df
)
# 15. TOPIC MODELING — TF-IDF
tfidf_vectorizer = TfidfVectorizer(

    stop_words=list(
        all_stopwords
    ),

    lowercase=True,

    ngram_range=(1, 2),

    min_df=2,

    max_df=0.80,

    max_features=3000,

    token_pattern=r"(?u)\b[a-zA-Z]{3,}\b"
)


tfidf_matrix = (
    tfidf_vectorizer
    .fit_transform(
        negative_reviews["clean_text"]
    )
)


tfidf_terms = (
    tfidf_vectorizer
    .get_feature_names_out()
)
# 16. NMF TOPIC MODEL

nmf = NMF(

    n_components=4,

    random_state=42,

    init="nndsvda",

    max_iter=2000
)


topic_matrix = (
    nmf.fit_transform(
        tfidf_matrix
    )
)
# 17. ASSIGN TOPIC TO EACH REVIEW

negative_reviews["topic_id"] = (
    topic_matrix
    .argmax(axis=1)
    + 1
)


negative_reviews["topic_strength"] = (
    topic_matrix
    .max(axis=1)
)
# 18. DISPLAY TOP WORDS OF EACH TOPIC
print("\n" + "=" * 70)
print("TOPIC MODELING RESULTS")
print("=" * 70)


topic_words = {}


for topic_idx, topic in enumerate(
    nmf.components_
):

    top_indices = (
        topic
        .argsort()[-15:][::-1]
    )

    words = [
        tfidf_terms[i]
        for i in top_indices
    ]

    topic_words[
        topic_idx + 1
    ] = words

    print(
        f"\nTOPIC {topic_idx + 1}"
    )

    print("-" * 70)

    print(
        ", ".join(words)
    )
# 19. AUTOMATIC TOPIC LABEL
topic_labels = {

    1: "Story & Characters",

    2: "Entertainment & Humor",

    3: "Acting & Pacing",

    4: "Controversial / Sensitive Themes"
}
# 20. Representative Reviews

for topic_id in sorted(negative_reviews["topic_id"].unique()):

    topic_name = topic_labels.get(
        topic_id,
        f"Topic {topic_id}"
    )

    print("\n" + "=" * 70)
    print(f"TOPIC {topic_id}: {topic_name}")
    print("=" * 70)

    topic_reviews = (
        negative_reviews[
            negative_reviews["topic_id"] == topic_id
        ]
        .sort_values("topic_strength", ascending=False)
        .drop_duplicates(subset=["name"])
        .head(5)
    )

    display(
        topic_reviews[
            [
                "name",
                "popularity",
                "user_rating",
                "topic_strength",
                "text"
            ]
        ]
    )
# 21. TOPIC SUMMARY

topic_summary = (

    negative_reviews

    .groupby("topic_id")

    .agg(

        Negative_Reviews=(
            "topic_id",
            "size"
        ),

        Number_of_Series=(
            "name",
            "nunique"
        )
    )

    .reset_index()
)
# 22. TOPIC PERCENTAGE

total_negative_reviews = (
    len(negative_reviews)
)


topic_summary["Percentage"] = (

    topic_summary[
        "Negative_Reviews"
    ]

    / total_negative_reviews

    * 100

).round(2)

# 23. TOPIC LABEL
topic_summary["Topic_Label"] = (

    topic_summary[
        "topic_id"
    ]

    .map(topic_labels)
)
# 24. SERIES COVERAGE

total_series = (

    negative_reviews[
        "name"
    ]

    .nunique()
)


topic_summary[
    "Series_Coverage_%"
] = (

    topic_summary[
        "Number_of_Series"
    ]

    / total_series

    * 100

).round(2)

# 25. POTENTIAL RED FLAG

topic_summary[
    "Potential_Red_Flag"
] = np.where(

    (

        (
            topic_summary[
                "Percentage"
            ] >= 20
        )

        &

        (
            topic_summary[
                "Number_of_Series"
            ] >= 2
        )

    ),

    "YES",

    "Review Further"
)

# 26. FINAL TOPIC TABLE

final_topic_table = (

    topic_summary[

        [
            "topic_id",

            "Topic_Label",

            "Negative_Reviews",

            "Percentage",

            "Number_of_Series",

            "Series_Coverage_%",

            "Potential_Red_Flag"
        ]
    ]

    .sort_values(

        "Percentage",

        ascending=False
    )

    .reset_index(
        drop=True
    )
)


final_topic_table.columns = [

    "Topic",

    "Topic_Label",

    "Negative_Reviews",

    "Percentage",

    "Number_of_Series",

    "Series_Coverage_%",

    "Potential_Red_Flag"
]


print("\n" + "=" * 70)
print("FINAL TOPIC SUMMARY & POTENTIAL RED FLAGS")
print("=" * 70)


display(
    final_topic_table
)


HIGH-POPULARITY THRESHOLD
80th Percentile Popularity = 192.8837
High-popularity reviews = 91

HIGH-POPULARITY SERIES WITH NEGATIVE REVIEWS
Negative reviews = 30
Number of series = 18


,name,popularity,user_rating,text
0,Law & Order: Special Victims Unit,566.1656,3.0,This series is a poster child for the way the ...
2,The Tonight Show Starring Jimmy Fallon,587.7775,2.0,"Fallon. What is there to say? He's not funny, ..."
5,Reacher,603.2777,5.0,## A tale of two Reachers\r\n### a matter of t...
10,Reacher,603.2777,4.0,SEASON 1: 8/10!\r\n\r\nDean even said it had z...
11,The Late Show with Stephen Colbert,564.1400,2.0,Stephen Colbert was once a funny man. Now he's...
15,The Simpsons,490.0595,4.0,Every now and again you find something hanging...
17,The Simpsons,490.0595,5.0,To address the failing quality:\r\n\r\nThe Sim...
23,Late Night with Seth Meyers,403.1640,1.0,"From Letterman to Seth, Late Night has lost it..."
25,The Office,406.9987,1.0,I will never understand the hype over this sho...
26,NCIS,340.4606,1.0,Written to depress young men.\r\nIt seems that...



CLEANING CHECK


,name,user_rating,clean_text
0,Law & Order: Special Victims Unit,3.0,poster child way wants world perceive justice ...
2,The Tonight Show Starring Jimmy Fallon,2.0,fallon funny entertaining seemingly reason bac...
5,Reacher,5.0,tale matter taste disclaimer read books cast c...
10,Reacher,4.0,dean zero wokery weird highlighted finlay race...
11,The Late Show with Stephen Colbert,2.0,stephen colbert funny man arrogant joke funny ...
15,The Simpsons,4.0,hanging fridge freezer forgot throw away feel ...
17,The Simpsons,5.0,address failing quality mocked mainstream pop ...
23,Late Night with Seth Meyers,1.0,letterman seth late night lost way seth boring...
25,The Office,1.0,understand hype premise absurd took years comp...
26,NCIS,1.0,written depress young men bellisario writes de...



TOP 20 KEYWORDS / N-GRAMS


...

,name,popularity,user_rating,topic_strength,text
79,Silo,292.1104,3.0,0.604894,Silo is pure progressive gospel in bunker form...
0,Law & Order: Special Victims Unit,566.1656,3.0,0.348127,This series is a poster child for the way the ...
27,Lioness,360.5732,2.0,0.281666,Amazed this has such a high rating. I managed ...
5,Reacher,603.2777,5.0,0.229564,## A tale of two Reachers\r\n### a matter of t...
114,Midsomer Murders,192.8837,1.0,0.135703,Spoiler Alert: Do you want to know who the kil...



TOPIC 4: Controversial / Sensitive Themes


,name,popularity,user_rating,topic_strength,text
63,Saturday Night Live,224.0310,1.0,0.722589,"It was good, then it was great, then it was go..."
17,The Simpsons,490.0595,5.0,0.528205,To address the failing quality:\r\n\r\nThe Sim...
35,Grey's Anatomy,377.8647,1.0,0.472206,"Yeah, it's like ER meets China Beach, and, as ..."
82,RuPaul's Drag Race,264.0714,1.0,0.187183,"Back in the 90s, RuPaul wasn't that bad. But ..."
39,Lanterns,279.6802,5.0,0.171307,Rated 5 at most. Disappointing. I thought this...



FINAL TOPIC SUMMARY & POTENTIAL RED FLAGS


,Topic,Topic_Label,Negative_Reviews,Percentage,Number_of_Series,Series_Coverage_%,Potential_Red_Flag
0,1,Story & Characters,11,36.67,7,38.89,YES
1,3,Acting & Pacing,10,33.33,6,33.33,YES
2,4,Controversial / Sensitive Themes,6,20.00,5,27.78,YES
3,2,Entertainment & Humor,3,10.00,3,16.67,Review Further


####
 กราฟ 3.3
